# **1. Instalação e Importação de Bibliotecas**

In [0]:
##1. Instalação das Bibliotecas

%pip install xgboost imbalanced-learn seaborn scikit-learn pandas numpy matplotlib --quiet

# Manipulação e Análise de Dados
import pandas as pd
import numpy as np

#Visualização
import matplotlib.pyplot as plt
import seaborn as sns

#Pré-Processamento
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE # Trata desbalanceamento de Classes

#Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

#Métricas de Avaliação
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
    f1_score,
    accuracy_score
)

#Configurações gerais
import warnings
warnings.filterwarnings ('ignore') #Suprime avisos desnecessários

sns.set_style('whitegrid') #Estilo visual dos gráficos
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42 # Semente para reprodutibilidade dos resultados

print('✅ Bibliotecas importadas com sucesso!')

# **2. Carregamento dos Dados**

In [0]:
#Deve fazer Upload do arquivo e nomear como "File_Name"
FILE_NAME = 'WineQT.csv'
df = pd.read_csv(FILE_NAME)

print(f'📊 Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas')
print('\n🔍 Primeiras 5 linhas:')
df.head()

# **3. Compreensão do Problema**

In [0]:
# A variável quality será transformada em classificação binária:
#1 = Alta qualidade - Nota >7
#0 = Baixa qualidade - Nota <7

#Verificação da distribuição original da variável 'quality'
#antes de transformá-la em binária
print('📊 Distribuição original da variável quality:')
print(df['quality'].value_counts().sort_index())

# Visualização
import os
os.makedirs('results', exist_ok=True)

plt.figure(figsize=(8, 4))
df['quality'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Distribuição original das notas de qualidade', fontsize=14)
plt.xlabel('Nota de Qualidade')
plt.ylabel('Quantidade de Amostras')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('results/distribuicao_quality_original.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
#Criação da variável alvo binária
#np.where funciona como umIF: se quality >=7, retorna 1; senão, retorna 0
df['quality_label'] = np.where(df['quality'] >=7, 1, 0)

# Contagem de cada classe após a transformação
contagem = df['quality_label'].value_counts()
proporcao = df['quality_label'].value_counts(normalize=True) * 100

print('🏷️ Distribuição da variável binária:')
print(f'  Alta qualidade (1):        {contagem[1]:>5} amostras ({proporcao[1]:.1f}%)')
print(f'  Baixa/Média qualidade (0): {contagem[0]:>5} amostras ({proporcao[0]:.1f}%)')

# Visualização do balanceamento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
contagem.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[0].set_title('Contagem por classe', fontsize=13)
axes[0].set_xticklabels(['Baixa/Média (0)', 'Alta (1)'], rotation=0)
axes[0].set_ylabel('Quantidade')

# Gráfico de pizza
proporcao.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
               colors=['#e74c3c', '#2ecc71'], labels=['Baixa/Média', 'Alta'])
axes[1].set_title('Proporção das classes', fontsize=13)
axes[1].set_ylabel('')

plt.suptitle('Balanceamento das Classes', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('results/balanceamento_classes.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# ⚠️ ATENÇÃO: Se uma classe tiver menos de ~30% do total,
# o dataset está desbalanceado e será necessário aplicar SMOTE
# (técnica de oversampling) — isso será tratado no pré-processamento
# ─────────────────────────────────────────────────────────────────

# **4. Análise Exploratória de Dados (EDA)**

In [0]:
#Visão geral do dataset: tipos de dados, valores nulos, estatísticas
print('='*60)
print('INFORMAÇÕES GERAIS DO DATASET')
print('='*60)
df.info()

print('\n' + '='*60)
print('VALORES NULOS POR COLUNA')
print('='*60)
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else '✅ Nenhum valor nulo encontrado!')

print('\n' + '='*60)
print('ESTATÍSTICAS DESCRITIVAS')
print('='*60)
df.describe().round(3)

In [0]:
# ─────────────────────────────────────────────────────────────────
# Distribuição de todas as variáveis numéricas
# Histogramas mostram como os valores se distribuem
# KDE (linha suave) mostra a densidade de probabilidade
# ─────────────────────────────────────────────────────────────────
features = [col for col in df.columns if col not in ['quality', 'quality_label', 'Id']]

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.histplot(df[feature], kde=True, ax=axes[i], color='steelblue', alpha=0.7)
    axes[i].set_title(f'Distribuição: {feature}', fontsize=11)
    axes[i].set_xlabel('')

# Remove subplots extras caso o número de features não preencha a grade
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribuição das Variáveis Físico-Químicas', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('results/distribuicao_variaveis.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────
# Boxplots por classe de qualidade
# Permitem comparar a distribuição de cada variável entre
# vinhos de alta qualidade (1) vs baixa/média qualidade (0)
# Os pontos fora das 'caixas' são outliers
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(features):
    sns.boxplot(
        data=df, x='quality_label', y=feature, ax=axes[i],
        palette={'0': '#e74c3c', '1': '#2ecc71'}
    )
    axes[i].set_title(f'{feature}', fontsize=11)
    axes[i].set_xlabel('Qualidade (0=Baixa/Média | 1=Alta)')

for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribuição das Variáveis por Classe de Qualidade', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('results/boxplots_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
💡 Interpretação:
  - Variáveis onde as caixas NÃO se sobrepõem muito têm MAIOR poder de separação entre as classes
  - Fique atento especialmente a 'alcohol', 'volatile acidity' e 'sulphates'
""")

In [0]:
# ─────────────────────────────────────────────────────────────────
# Mapa de correlação (Heatmap)
# Correlação mede a relação linear entre duas variáveis:
#   +1 = correlação positiva perfeita
#    0 = sem correlação
#   -1 = correlação negativa perfeita
# ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(13, 10))

corr_matrix = df[features + ['quality_label']].corr()

mascara = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Exibe apenas metade do heatmap

sns.heatmap(
    corr_matrix,
    mask=mascara,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    square=True
)
plt.title('Mapa de Correlação entre Variáveis', fontsize=15)
plt.tight_layout()
plt.savefig('results/heatmap_correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

# ─────────────────────────────────────────────────────────────────
# Correlação de cada variável com a variável alvo
# ─────────────────────────────────────────────────────────────────
print('📊 Correlação de cada variável com quality_label (ordenada):')
corr_target = corr_matrix['quality_label'].drop('quality_label').sort_values(ascending=False)
print(corr_target.round(3).to_string())

print("""
💡 Interpretação das correlações com quality_label:
  - Valores positivos: maior valor da variável → maior chance de vinho ser de alta qualidade
  - Valores negativos: maior valor da variável → menor chance de vinho ser de alta qualidade
  - Valores próximos de 0: pouca relação linear com a qualidade
""")

# **5. Limpeza dos Dados**

In [0]:
# ─────────────────────────────────────────────────────────────────
# ETAPA 1 — Verificação e remoção de duplicatas
# Linhas duplicadas são registros idênticos em todas as colunas.
# Diferente dos outliers, duplicatas genuinamente não agregam
# informação nova e podem inflar artificialmente o desempenho do modelo
# ─────────────────────────────────────────────────────────────────
print('=' * 55)
print('VERIFICAÇÃO DE DUPLICATAS')
print('=' * 55)

n_antes = df.shape[0]
duplicatas = df.duplicated().sum()
print(f'Total de linhas antes da limpeza: {n_antes}')
print(f'Linhas duplicadas encontradas:    {duplicatas}')

if duplicatas > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'✅ {duplicatas} duplicata(s) removida(s)!')
    print(f'Total de linhas após remoção:     {df.shape[0]}')
else:
    print('✅ Nenhuma duplicata encontrada!')

In [0]:
# ─────────────────────────────────────────────────────────────────
# ETAPA 2 — Verificação e tratamento de valores nulos
# Valores nulos (NaN) precisam ser tratados antes da modelagem
# pois algoritmos de ML não aceitam dados ausentes.
# Estratégia adotada:
#   - Se < 5% da coluna for nulo → preenche com a mediana
#     (mediana é mais robusta que a média quando há outliers)
#   - Se >= 5% → remove as linhas afetadas
# ─────────────────────────────────────────────────────────────────
print('=' * 55)
print('VERIFICAÇÃO DE VALORES NULOS')
print('=' * 55)

nulos = df.isnull().sum()
nulos_pct = (nulos / len(df)) * 100

resumo_nulos = pd.DataFrame({
    'Nulos': nulos,
    '% do Total': nulos_pct.round(2)
})
resumo_nulos = resumo_nulos[resumo_nulos['Nulos'] > 0]

if resumo_nulos.empty:
    print('✅ Nenhum valor nulo encontrado! Dataset limpo.')
else:
    print(resumo_nulos.to_string())
    print()
    for col in resumo_nulos.index:
        pct = nulos_pct[col]
        if pct < 5:
            mediana = df[col].median()
            df[col].fillna(mediana, inplace=True)
            print(f'  → "{col}": {pct:.1f}% nulo — preenchido com mediana ({mediana:.3f})')
        else:
            n_removidas = df[col].isnull().sum()
            df = df.dropna(subset=[col])
            print(f'  → "{col}": {pct:.1f}% nulo — {n_removidas} linhas removidas')
    print(f'\n✅ Tratamento concluído! Total de linhas: {df.shape[0]}')

In [0]:
# ─────────────────────────────────────────────────────────────────
# ETAPA 3 — Análise de outliers (sem remoção)
#
# Utilizamos o método IQR para QUANTIFICAR os outliers por variável:
#   Q1 = 25º percentil | Q3 = 75º percentil | IQR = Q3 - Q1
#   Outlier convencional: fora de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
#
# DECISÃO: os outliers NÃO serão removidos.
# Justificativa: este dataset contém medições reais de vinhos.
# Valores extremos de acidez, sulfatos ou álcool representam vinhos
# reais que existem no mercado — e precisam ser corretamente
# classificados pelo modelo. Removê-los seria:
#   1. Perda de informação relevante para o negócio
#   2. Redução do dataset já pequeno (~1600 registros)
#   3. Criação de um modelo que funciona apenas em condições ideais
# Os modelos escolhidos (Random Forest e XGBoost) são baseados em
# árvores de decisão, que são naturalmente robustos a outliers.
# ─────────────────────────────────────────────────────────────────

features_num = [col for col in df.columns if col not in ['quality', 'quality_label', 'Id']]

relatorio_outliers = []
for col in features_num:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    n_outliers = ((df[col] < limite_inf) | (df[col] > limite_sup)).sum()
    pct_outliers = (n_outliers / len(df)) * 100
    relatorio_outliers.append({
        'Variável':    col,
        'Q1':          round(Q1, 3),
        'Q3':          round(Q3, 3),
        'Limite Inf':  round(limite_inf, 3),
        'Limite Sup':  round(limite_sup, 3),
        'Outliers':    n_outliers,
        '% Outliers':  round(pct_outliers, 2)
    })

df_outliers = pd.DataFrame(relatorio_outliers).set_index('Variável')

print('=' * 65)
print('RELATÓRIO DE OUTLIERS POR VARIÁVEL (método IQR x1.5)')
print('=' * 65)
print(df_outliers.to_string())
print(f'\n⚠️  Outliers identificados e documentados — mantidos no dataset.')
print(f'✅ Total de registros preservados: {df.shape[0]}')

In [0]:
# ─────────────────────────────────────────────────────────────────
# Visualização dos outliers por variável
# Os pontos além dos 'bigodes' do boxplot são os outliers.
# Separamos por classe (alta vs baixa qualidade) para entender
# se os outliers estão concentrados em algum grupo específico
# ─────────────────────────────────────────────────────────────────

# Top 6 variáveis com mais outliers
top_vars = (
    df_outliers['Outliers']
    .sort_values(ascending=False)
    .head(6)
    .index
    .tolist()
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(top_vars):
    n_out = df_outliers.loc[col, 'Outliers']
    pct   = df_outliers.loc[col, '% Outliers']
    sns.boxplot(
        data=df, x='quality_label', y=col, ax=axes[i],
        palette={'0': '#e74c3c', '1': '#2ecc71'}
    )
    axes[i].set_title(f'{col}\n({n_out} outliers | {pct}% do total)', fontsize=11)
    axes[i].set_xlabel('0 = Baixa/Média  |  1 = Alta Qualidade')

plt.suptitle(
    'Outliers por Variável e Classe de Qualidade\n'
    '(mantidos no dataset — representam vinhos reais)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('results/analise_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n💡 Interpretação:')
print('  Os outliers são valores extremos de medições reais.')
print('  Removê-los significaria descartar vinhos com características')
print('  incomuns que o modelo precisa aprender a classificar.')
print('  Os modelos baseados em árvores (Random Forest, XGBoost)')
print('  lidam naturalmente bem com essa variabilidade.')
print(f'\n✅ Limpeza concluída! Dataset final: {df.shape[0]} registros.')

In [0]:
# ─────────────────────────────────────────────────────────────────
# Resumo final da limpeza
# ─────────────────────────────────────────────────────────────────
print('=' * 55)
print('RESUMO DA LIMPEZA DE DADOS')
print('=' * 55)
print(f'  Registros originais:          {n_antes}')
print(f'  Duplicatas removidas:         {duplicatas}')
print(f'  Nulos tratados:               {nulos.sum() if nulos.sum() > 0 else 0}')
print(f'  Outliers removidos:           0 (decisão de negócio)')
print(f'  Registros finais:             {df.shape[0]}')
print('=' * 55)

# **6. Pré-processamento de Dados**

In [0]:
# ─────────────────────────────────────────────────────────────────
# Separação entre features (X) e variável alvo (y)
# X = todas as colunas que o modelo vai usar para aprender
# y = o que o modelo precisa prever (alta ou baixa qualidade)
# ─────────────────────────────────────────────────────────────────

# Remove colunas que não são features úteis
colunas_remover = ['quality', 'quality_label']
if 'Id' in df.columns:
    colunas_remover.append('Id')  # Coluna de ID não tem valor preditivo

X = df.drop(columns=colunas_remover)
y = df['quality_label']

print(f'✅ Features (X): {X.shape[1]} variáveis, {X.shape[0]} amostras')
print(f'✅ Target  (y): {y.shape[0]} amostras')
print(f'\nVariáveis usadas: {list(X.columns)}')

In [0]:
# ─────────────────────────────────────────────────────────────────
# Divisão em treino e teste
# O modelo aprende com os dados de TREINO (80%)
# e é avaliado nos dados de TESTE (20%) — que nunca viu antes
# stratify=y garante que a proporção de classes seja mantida nos dois conjuntos
# ─────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'📂 Tamanho do conjunto de treino: {X_train.shape[0]} amostras')
print(f'📂 Tamanho do conjunto de teste:  {X_test.shape[0]} amostras')

In [0]:
# ─────────────────────────────────────────────────────────────────
# Normalização (StandardScaler)
# Transforma cada variável para ter média 0 e desvio padrão 1
# Isso é importante porque variáveis em escalas muito diferentes
# (ex: alcohol em % vs SO2 em mg/L) podem distorcer o aprendizado
# IMPORTANTE: o scaler é 'fitado' APENAS no treino, para evitar
# vazamento de informação (data leakage)
# ─────────────────────────────────────────────────────────────────
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # Aprende a escala + transforma
X_test_scaled  = scaler.transform(X_test)         # Aplica a MESMA escala do treino

print('✅ Normalização aplicada com StandardScaler')

In [0]:
# ─────────────────────────────────────────────────────────────────
# Tratamento do desbalanceamento com SMOTE
# SMOTE (Synthetic Minority Over-sampling Technique) gera amostras
# sintéticas da classe minoritária para equilibrar o dataset
# Isso evita que o modelo 'ignore' a classe menos frequente
# SMOTE é aplicado APENAS no conjunto de treino!
# ─────────────────────────────────────────────────────────────────
print('Distribuição ANTES do SMOTE:')
print(pd.Series(y_train).value_counts().to_string())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

print('\nDistribuição APÓS o SMOTE:')
print(pd.Series(y_train_bal).value_counts().to_string())
print('\n✅ Classes balanceadas com SMOTE!')

# **6. Desenvolvimento dos Modelos**

In [0]:
# ─────────────────────────────────────────────────────────────────
# Definição dos modelos
# Cada modelo é instanciado com hiperparâmetros padrão + random_state
# para garantir reprodutibilidade dos resultados
# ─────────────────────────────────────────────────────────────────
modelos = {
    'Regressão Logística':  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'XGBoost':              XGBClassifier(n_estimators=100, use_label_encoder=False,
                                          eval_metric='logloss', random_state=RANDOM_STATE)
}

# ─────────────────────────────────────────────────────────────────
# Treinamento de todos os modelos
# .fit() é o método que executa o aprendizado
# ─────────────────────────────────────────────────────────────────
modelos_treinados = {}

for nome, modelo in modelos.items():
    print(f'⏳ Treinando: {nome}...')
    modelo.fit(X_train_bal, y_train_bal)
    modelos_treinados[nome] = modelo
    print(f'✅ {nome} treinado!')

print('\n🎉 Todos os modelos foram treinados!')

# **7. Avaliação e Comparação dos Modelos**

In [0]:
# ─────────────────────────────────────────────────────────────────
# Avaliação de cada modelo no conjunto de teste
# Geramos previsões e calculamos todas as métricas relevantes
# ─────────────────────────────────────────────────────────────────
resultados = []

for nome, modelo in modelos_treinados.items():
    y_pred      = modelo.predict(X_test_scaled)          # Previsões de classe (0 ou 1)
    y_pred_prob = modelo.predict_proba(X_test_scaled)[:,1]  # Probabilidade da classe 1

    resultados.append({
        'Modelo':    nome,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_pred_prob),
    })

# Cria tabela comparativa ordenada pelo F1-Score
df_resultados = pd.DataFrame(resultados).sort_values('F1-Score', ascending=False)
df_resultados = df_resultados.set_index('Modelo')
df_resultados = df_resultados.round(4)

print('='*60)
print('COMPARAÇÃO DE DESEMPENHO DOS MODELOS')
print('='*60)
print(df_resultados.to_string())
print(f'\n🏆 Melhor modelo por F1-Score: {df_resultados["F1-Score"].idxmax()}')

In [0]:
# ─────────────────────────────────────────────────────────────────
# Visualização comparativa das métricas
# ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

df_resultados.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.7)
ax.set_title('Comparação de Métricas por Modelo', fontsize=15)
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Linha de referência 0.8')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('results/comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────
# Matrizes de Confusão para todos os modelos
# A matriz mostra:
#   - Verdadeiros Positivos (TP): acertou 'alta qualidade'
#   - Verdadeiros Negativos (TN): acertou 'baixa/média qualidade'
#   - Falsos Positivos (FP): classificou como alta, mas era baixa
#   - Falsos Negativos (FN): classificou como baixa, mas era alta
# ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (nome, modelo) in enumerate(modelos_treinados.items()):
    y_pred = modelo.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Baixa/Média', 'Alta'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{nome}', fontsize=12)

plt.suptitle('Matrizes de Confusão', fontsize=16)
plt.tight_layout()
plt.savefig('results/matrizes_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────
# Curvas ROC
# Mostra o trade-off entre Taxa de Verdadeiros Positivos (TPR)
# e Taxa de Falsos Positivos (FPR) em diferentes limiares de decisão
# Quanto maior a área sob a curva (AUC), melhor o modelo
# ─────────────────────────────────────────────────────────────────
plt.figure(figsize=(9, 6))

cores = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for (nome, modelo), cor in zip(modelos_treinados.items(), cores):
    y_pred_prob = modelo.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
    auc = roc_auc_score(y_test, y_pred_prob)
    plt.plot(fpr, tpr, label=f'{nome} (AUC = {auc:.3f})', color=cor, lw=2)

# Linha diagonal = modelo aleatório (AUC = 0.5)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatório (AUC = 0.50)')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos')
plt.title('Curvas ROC — Comparação dos Modelos', fontsize=14)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('results/curvas_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# ─────────────────────────────────────────────────────────────────
# Relatório detalhado do melhor modelo
# ─────────────────────────────────────────────────────────────────
melhor_nome  = df_resultados['F1-Score'].idxmax()
melhor_modelo = modelos_treinados[melhor_nome]
y_pred_melhor = melhor_modelo.predict(X_test_scaled)

print(f'🏆 Relatório detalhado do melhor modelo: {melhor_nome}')
print('='*60)
print(classification_report(y_test, y_pred_melhor,
                              target_names=['Baixa/Média', 'Alta Qualidade']))

# **8. Interpretação dos Resultados**

In [0]:
# ─────────────────────────────────────────────────────────────────
# Feature Importance — importância de cada variável para o modelo
# Modelos baseados em árvores (Random Forest, XGBoost) fornecem
# diretamente a importância de cada feature no processo decisório
# ─────────────────────────────────────────────────────────────────

# Pegamos o Random Forest e o XGBoost para comparar as importâncias
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

modelos_tree = ['Random Forest', 'XGBoost']

for ax, nome in zip(axes, modelos_tree):
    modelo = modelos_treinados[nome]
    importancias = pd.Series(modelo.feature_importances_, index=X.columns)
    importancias = importancias.sort_values(ascending=True)

    importancias.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(f'Feature Importance — {nome}', fontsize=13)
    ax.set_xlabel('Importância')

plt.suptitle('Variáveis mais Influentes na Qualidade do Vinho', fontsize=15)
plt.tight_layout()
plt.savefig('results/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Ranking textual
print('📊 Top 5 variáveis mais importantes (Random Forest):')
rf_imp = pd.Series(
    modelos_treinados['Random Forest'].feature_importances_,
    index=X.columns
).sort_values(ascending=False)

for i, (feat, val) in enumerate(rf_imp.head(5).items(), 1):
    print(f'  {i}. {feat}: {val:.4f}')

# 9. Conclusão
# Resumo dos Resultados

In [0]:
# ─────────────────────────────────────────────────────────────────
# Resumo final — imprime um relatório consolidado do projeto
# ─────────────────────────────────────────────────────────────────
melhor = df_resultados.iloc[0]

print('=' * 65)
print('  RELATÓRIO FINAL — CLASSIFICAÇÃO DE QUALIDADE DE VINHOS')
print('=' * 65)
print(f'\n📦 Dataset: {df.shape[0]} amostras | {X.shape[1]} features')
print(f'🎯 Critério de alta qualidade: nota ≥ 7')
print(f'⚖️  Balanceamento: SMOTE aplicado no conjunto de treino')
print(f'\n🏆 Melhor modelo: {df_resultados["F1-Score"].idxmax()}')
print(f'   Accuracy : {melhor["Accuracy"]:.4f}')
print(f'   F1-Score : {melhor["F1-Score"]:.4f}')
print(f'   ROC-AUC  : {melhor["ROC-AUC"]:.4f}')

print('\n📊 Comparação completa de modelos:')
print(df_resultados.to_string())

print('\n💡 Principais variáveis preditoras (Random Forest):')
for i, (feat, val) in enumerate(rf_imp.head(5).items(), 1):
    print(f'   {i}. {feat}: {val:.4f}')

print('\n🍷 Implicações para o processo produtivo:')
print('   Os resultados indicam que as variáveis físico-químicas')
print('   mais relevantes podem ser monitoradas e ajustadas durante')
print('   a produção para aumentar a probabilidade de um vinho')
print('   atingir alta qualidade, reduzindo a dependência exclusiva')
print('   da avaliação sensorial por especialistas.')
print('=' * 65)

In [0]:
# ─────────────────────────────────────────────────────────────────
# Salva a tabela de resultados em CSV para o repositório
# ─────────────────────────────────────────────────────────────────
import os
os.makedirs('results', exist_ok=True)  # Cria a pasta se não existir

df_resultados.to_csv('results/comparacao_modelos.csv')
print('✅ Resultados salvos em results/comparacao_modelos.csv')
print('✅ Gráficos salvos na pasta results/')